# Model 1 (Sederhana) — Binary Logistic Regression
## Pemilihan Hotel: Novotel (pasca-merger) vs Pullman

Notebook ini membangun **model logit sederhana** (belum melibatkan efek interaksi) untuk
memprediksi pilihan tamu antara **Novotel** dan **Pullman**, mulai dari estimasi hingga
validasi hold-out. Efek interaksi (Income x TujuanMenginap) akan dibahas terpisah sebagai
**Model 2** untuk dibandingkan dengan model ini (poin B.7 di soal).

**Definisi variabel dependen:** `BrandPilihan_Pullman` = 1 jika tamu memilih **Pullman**,
0 jika memilih **Novotel** (Novotel jadi kategori referensi/basis).


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from logit_diagnostics import full_diagnostic_report, validation_report, train_fit_report, train_test_comparison


## 1. Load Data

In [2]:
df = pd.read_csv('Data_Binary_Logit_Pullman-vs-Novotel.csv')
print(df.shape)
df.head()


(500, 8)


,RespID,TujuanMenginap,Income_JutaRp,Harga_Novotel,Harga_Pullman,Iklan_Novotel,Iklan_Pullman,BrandPilihan
0,1,Bisnis,16.7,738000,1207000,0,0,Novotel
1,2,Bisnis,11.2,626000,1615000,0,1,Pullman
2,3,Leisure/Pribadi,20.6,911000,1301000,0,0,Novotel
3,4,Bisnis,3.0,914000,1204000,0,0,Pullman
4,5,Leisure/Pribadi,17.3,1100000,1837000,1,1,Novotel


In [3]:
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
RespID,500.0,NaN,NaN,NaN,250.5,144.481833,1.0,125.75,250.5,375.25,500.0
TujuanMenginap,500,2,Leisure/Pribadi,259,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Income_JutaRp,500.0,NaN,NaN,NaN,15.0548,9.317153,3.0,6.95,14.55,22.2,52.5
Harga_Novotel,500.0,NaN,NaN,NaN,851336.0,144009.487775,600000.0,723000.0,856500.0,975000.0,1100000.0
Harga_Pullman,500.0,NaN,NaN,NaN,1594392.0,226642.166596,1204000.0,1389750.0,1594500.0,1778250.0,1997000.0
Iklan_Novotel,500.0,NaN,NaN,NaN,0.188,0.391103,0.0,0.0,0.0,0.0,1.0
Iklan_Pullman,500.0,NaN,NaN,NaN,0.212,0.409134,0.0,0.0,0.0,0.0,1.0
BrandPilihan,500,2,Pullman,254,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Operasionalisasi Variabel

- **Variabel Dependen (Y):** `BrandPilihan_Pullman` — 1 = Pullman, 0 = Novotel (referensi)
- **Variabel Independen (X):**
  - `TujuanMenginap_Bisnis` — dummy, 1 = Bisnis, 0 = Leisure/Pribadi (referensi)
  - `Income_JutaRp` — pendapatan responden (juta Rp/bulan), kontinu
  - `Harga_Novotel`, `Harga_Pullman` — harga kamar rata-rata masing-masing brand (Rp/malam)
  - `Iklan_Novotel`, `Iklan_Pullman` — status kampanye iklan brand tsb (1=ya, 0=tidak)

Model 1 ini adalah spesifikasi **tanpa efek interaksi** — asumsi sementara bahwa pengaruh
setiap variabel terhadap pilihan brand konstan untuk semua tamu.


In [4]:
df_model = df.copy()

# Dummy variabel kategorikal
df_model['BrandPilihan_Pullman'] = (df_model['BrandPilihan'] == 'Pullman').astype(int)
df_model['TujuanMenginap_Bisnis'] = (df_model['TujuanMenginap'] == 'Bisnis').astype(int)

feature_cols = [
    'TujuanMenginap_Bisnis',
    'Income_JutaRp',
    'Harga_Novotel',
    'Harga_Pullman',
    'Iklan_Novotel',
    'Iklan_Pullman',
]

X = df_model[feature_cols]
y = df_model['BrandPilihan_Pullman']

print('Proporsi Y=1 (Pullman):', y.mean().round(3))
df_model[['BrandPilihan', 'BrandPilihan_Pullman', 'TujuanMenginap', 'TujuanMenginap_Bisnis'] + feature_cols[1:]].head()


Proporsi Y=1 (Pullman): 0.508


,BrandPilihan,BrandPilihan_Pullman,TujuanMenginap,TujuanMenginap_Bisnis,Income_JutaRp,Harga_Novotel,Harga_Pullman,Iklan_Novotel,Iklan_Pullman
0,Novotel,0,Bisnis,1,16.7,738000,1207000,0,0
1,Pullman,1,Bisnis,1,11.2,626000,1615000,0,1
2,Novotel,0,Leisure/Pribadi,0,20.6,911000,1301000,0,0
3,Pullman,1,Bisnis,1,3.0,914000,1204000,0,0
4,Novotel,0,Leisure/Pribadi,0,17.3,1100000,1837000,1,1


## 3. Estimasi Model (80% Data Training)

Split data secara acak: 80% untuk estimasi (training), 20% untuk validasi (hold-out),
menggunakan `random_state` tetap agar hasil dapat direplikasi.


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, ' | Test:', X_test.shape)
print('Proporsi Y=1 di Train:', y_train.mean().round(3), ' | di Test:', y_test.mean().round(3))


Train: (400, 6)  | Test: (100, 6)
Proporsi Y=1 di Train: 0.507  | di Test: 0.51


## 4. Full Diagnostic Report — Model 1 (Training)

Mencakup: korelasi awal, koefisien & Wald test, Likelihood Ratio test, McFadden Pseudo R²,
Hosmer-Lemeshow test, VIF, classification table, dan Press's Q — semua pada data training.


In [6]:
model1, results1 = full_diagnostic_report(
    X_train, y_train,
    feature_names=feature_cols,
    y_name='BrandPilihan_Pullman',
    model_label='Model 1 (Simple, No Interaction)',
    labels=('Novotel', 'Pullman'),
)



######################################################################
#  FULL DIAGNOSTIC REPORT (Binary Logit) - Model 1 (Simple, No Interaction)
######################################################################

1. PEARSON CORRELATION MATRIX (antar variabel independen)
                       TujuanMenginap_Bisnis  Income_JutaRp  Harga_Novotel  Harga_Pullman  Iklan_Novotel  Iklan_Pullman
TujuanMenginap_Bisnis                  1.000          0.050         -0.037          0.000          0.088          0.037
Income_JutaRp                          0.050          1.000         -0.001          0.017          0.043         -0.001
Harga_Novotel                         -0.037         -0.001          1.000          0.023         -0.023          0.072
Harga_Pullman                          0.000          0.017          0.023          1.000          0.115         -0.065
Iklan_Novotel                          0.088          0.043         -0.023          0.115          1.000          0.035
Ik

## 5. Ringkasan statsmodels (opsional, untuk cross-check)

In [7]:
X_train_const = sm.add_constant(X_train)
print(model1.summary())


                            Logit Regression Results                            
Dep. Variable:     BrandPilihan_Pullman   No. Observations:                  400
Model:                            Logit   Df Residuals:                      393
Method:                             MLE   Df Model:                            6
Date:                  Sat, 12 Sep 2026   Pseudo R-squ.:                  0.2648
Time:                          10:29:34   Log-Likelihood:                -203.80
converged:                         True   LL-Null:                       -277.21
Covariance Type:              nonrobust   LLR p-value:                 3.612e-29
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -2.0085      1.125     -1.786      0.074      -4.213       0.196
TujuanMenginap_Bisnis     2.0377      0.262      7.777      0.000       1.524     

## 6. Fit di Data Training (in-sample) vs Validasi Hold-Out (20%)

Bandingkan performa klasifikasi pada data yang dipakai untuk estimasi (training) dengan
performa pada data yang belum pernah dilihat model (test/hold-out) untuk mengecek indikasi
overfitting.


In [8]:
train_result = train_fit_report(model1, X_train, y_train, label='Training Set (80%)',
                                 labels=('Novotel', 'Pullman'))


FIT MODEL DI DATA TRAINING - Training Set (80%)
--- Classification Table (threshold = 0.5) ---
                Pred Novotel  Pred Pullman
Actual Novotel           146            51
Actual Pullman            43           160

N                  : 400
Hit Rate (Akurasi) : 0.7650  (76.50%)
Sensitivity (Recall Pullman) : 0.7882
Specificity (Recall Novotel) : 0.7411
Precision (Pullman)          : 0.7583

Proportional Chance Criterion : 0.5001 (hit rate model sebaiknya > 1.25x ini = 0.6251, aturan Hair et al.)
  -> Model MEMENUHI kriteria (>1.25x chance)

--- Press's Q Statistic ---
Press's Q = 112.3600  (nilai kritis chi2(1, 0.05) = 3.8415, p-value = 0.0000)
  -> Signifikan (Q > 3.841): hit rate model JAUH LEBIH BAIK dari tebakan acak



In [9]:
test_result = validation_report(model1, X_test, y_test, label='Hold-Out / Test Set (20%)',
                                 labels=('Novotel', 'Pullman'))


VALIDASI MODEL - Hold-Out / Test Set (20%)
--- Classification Table (threshold = 0.5) ---
                Pred Novotel  Pred Pullman
Actual Novotel            35            14
Actual Pullman            14            37

N                  : 100
Hit Rate (Akurasi) : 0.7200  (72.00%)
Sensitivity (Recall Pullman) : 0.7255
Specificity (Recall Novotel) : 0.7143
Precision (Pullman)          : 0.7255

Proportional Chance Criterion : 0.5002 (hit rate model sebaiknya > 1.25x ini = 0.6252, aturan Hair et al.)
  -> Model MEMENUHI kriteria (>1.25x chance)

--- Press's Q Statistic ---
Press's Q = 19.3600  (nilai kritis chi2(1, 0.05) = 3.8415, p-value = 0.0000)
  -> Signifikan (Q > 3.841): hit rate model JAUH LEBIH BAIK dari tebakan acak



In [10]:
train_test_comparison(train_result, test_result)


RINGKASAN PERBANDINGAN TRAIN vs TEST
Hit Rate         Train      Test         Gap
                0.7650    0.7200     +0.0450



## 7. Interpretasi Odds Ratio ke Bahasa Bisnis

Odds Ratio (OR) menunjukkan seberapa besar odds (peluang relatif) memilih **Pullman**
(dibanding Novotel) berubah untuk setiap kenaikan 1 unit variabel independen, dengan
variabel lain dianggap tetap (ceteris paribus).


In [11]:
or_table = np.exp(model1.params).rename('Odds Ratio').to_frame()
or_table['Perubahan Odds (%)'] = (or_table['Odds Ratio'] - 1) * 100
or_table['p-value'] = model1.pvalues
or_table['Signifikan (5%)'] = or_table['p-value'] < 0.05
or_table.round(4)


,Odds Ratio,Perubahan Odds (%),p-value,Signifikan (5%)
const,0.1342,-86.5812,0.0741,False
TujuanMenginap_Bisnis,7.6728,667.2800,0.0000,True
Income_JutaRp,1.1210,12.0974,0.0000,True
Harga_Novotel,1.0000,0.0001,0.2609,False
Harga_Pullman,1.0000,-0.0001,0.1386,False
Iklan_Novotel,0.3459,-65.4138,0.0011,True
Iklan_Pullman,1.5023,50.2334,0.2016,False


In [12]:
print('Contoh interpretasi otomatis (variabel signifikan saja):\n')
for var in feature_cols:
    p = model1.pvalues[var]
    if p < 0.05:
        or_val = np.exp(model1.params[var])
        pct = (or_val - 1) * 100
        arah = 'menaikkan' if or_val > 1 else 'menurunkan'
        print(f"- {var}: signifikan (p={p:.4f}). Setiap kenaikan 1 unit {arah} odds "
              f"memilih Pullman sebesar {abs(pct):.1f}% (OR={or_val:.3f}), ceteris paribus.")
    else:
        print(f"- {var}: TIDAK signifikan (p={p:.4f}) pada Model 1 ini.")


Contoh interpretasi otomatis (variabel signifikan saja):

- TujuanMenginap_Bisnis: signifikan (p=0.0000). Setiap kenaikan 1 unit menaikkan odds memilih Pullman sebesar 667.3% (OR=7.673), ceteris paribus.
- Income_JutaRp: signifikan (p=0.0000). Setiap kenaikan 1 unit menaikkan odds memilih Pullman sebesar 12.1% (OR=1.121), ceteris paribus.
- Harga_Novotel: TIDAK signifikan (p=0.2609) pada Model 1 ini.
- Harga_Pullman: TIDAK signifikan (p=0.1386) pada Model 1 ini.
- Iklan_Novotel: signifikan (p=0.0011). Setiap kenaikan 1 unit menurunkan odds memilih Pullman sebesar 65.4% (OR=0.346), ceteris paribus.
- Iklan_Pullman: TIDAK signifikan (p=0.2016) pada Model 1 ini.


## 8. Catatan Lanjutan

Model 1 di atas adalah **spesifikasi sederhana tanpa efek interaksi**. Langkah selanjutnya
(di luar cakupan notebook ini) adalah membangun **Model 2** dengan menambahkan interaksi
`Income_JutaRp x TujuanMenginap_Bisnis` — sesuai hint pada soal bahwa sensitivitas income
terhadap pilihan brand bisa berbeda antara tamu bisnis (biaya di-reimburse kantor) vs
leisure/pribadi (biaya ditanggung sendiri) — lalu membandingkan Model 1 vs Model 2 dengan
Likelihood Ratio test antar-model serta AIC/BIC (poin B.7).
